# Preprocesamiento y normalización

**PP2 · ITSE** — Input: `viviendas_sinteticas.csv` | Output: `viviendas_procesadas.csv`

Transforma el dataset crudo en un formato listo para análisis y modelado. Cada transformación está justificada y documentada.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scikit-learn -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
DRIVE = Path('/content/drive/MyDrive/PP2')
Path('img').mkdir(exist_ok=True)

df_raw = pd.read_csv(DRIVE / 'viviendas_sinteticas.csv')
df = df_raw.copy()
col_avance = 'avance_obra' if 'avance_obra' in df.columns else 'avanceObra'
col_tipo   = 'tipo_vivienda' if 'tipo_vivienda' in df.columns else 'tipoVivienda'
col_dorm   = 'cant_dormitorios' if 'cant_dormitorios' in df.columns else 'cantDormitorios'
print(f'Dataset cargado — {len(df)} registros, {df.shape[1]} columnas')

In [ ]:
# 1. Fechas → datetime + variables derivadas
col_fi = 'fecha_inic' if 'fecha_inic' in df.columns else 'fechaInic'
col_ff = 'fecha_fin'  if 'fecha_fin'  in df.columns else 'fechaFin'

# errors='coerce' convierte fechas inválidas a NaT en lugar de tirar un error.
# Se prefiere marcarlas sobre eliminar la fila: el resto de los campos sigue siendo válido.
df['fecha_inic_dt'] = pd.to_datetime(df[col_fi], format='%d-%m-%Y', errors='coerce')
df['fecha_fin_dt']  = pd.to_datetime(df[col_ff], format='%d-%m-%Y', errors='coerce')

hoy = pd.Timestamp.today()
# Para obras activas (sin fecha_fin), usamos hoy como fecha de corte: la obra sigue corriendo.
df['dias_activa'] = (df['fecha_fin_dt'].fillna(hoy) - df['fecha_inic_dt']).dt.days
df['anio_inicio'] = df['fecha_inic_dt'].dt.year

print(f'Fechas inválidas: {df["fecha_inic_dt"].isna().sum()}')
print(f'Rango: {df["fecha_inic_dt"].min().date()} → {df["fecha_inic_dt"].max().date()}')
print(df[['dias_activa','anio_inicio']].describe().round(1))

In [ ]:
# 2. Detección de inconsistencias
mask = (
    ((df['estado'] == 'Finalizada') & (df[col_avance] < 80)) |
    ((df['estado'] == 'Iniciada')   & (df[col_avance] > 60)) |
    (df['fecha_fin_dt'] < df['fecha_inic_dt'])
)
df['inconsistente'] = mask
n = mask.sum()
print(f'Registros inconsistentes: {n} ({n/len(df)*100:.1f}%)')
if n > 0:
    display(df[mask][['num_exp','estado',col_avance]].head(10))

# No se eliminan: se marcan con una columna booleana para informar al equipo de Programación.
# Eliminarlos ocultaría errores sistemáticos de carga en el sistema VIVSO que deberían corregirse
# en la fuente, no silenciarse en el análisis.

In [ ]:
# 3. Encoding de variables categóricas
# criterio se encoda manualmente para preservar el orden lógico del programa:
# Inclusion (0) → apta | Otro (1) → caso especial | Exclusion (2) → rechazada.
# Con LabelEncoder el orden sería alfabético (Exclusion=0, Inclusion=1, Otro=2),
# que no refleja la jerarquía del programa y haría los modelos menos interpretables.
criterio_map = {'Inclusion': 0, 'Otro': 1, 'Exclusion': 2}
df['criterio_enc'] = df['criterio'].map(criterio_map)

encoders = {}
for col in ['estado', 'clasificacion', col_tipo]:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].fillna('Desconocido'))
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('Mapeos de encoding:')
for col, m in encoders.items():
    print(f'  {col}: {m}')
print(f'  criterio: {criterio_map}')

In [ ]:
# 4. Normalización de variables numéricas a [0, 1]
cols_num = [col_avance, 'superficie', col_dorm, 'dias_activa']
for col in cols_num:
    if df[col].isna().any():
        med = df[col].median()
        df[col] = df[col].fillna(med)

# MinMaxScaler es necesario porque las variables tienen escalas muy distintas:
# avance_obra (0–100), superficie (~30–60 m²), dias_activa (0–600+).
# Sin normalizar, dias_activa dominaría cualquier modelo de ML solo por su magnitud,
# no por su relevancia real. [0,1] le da el mismo peso inicial a cada variable.
scaler = MinMaxScaler()
df[[f'{c}_norm' for c in cols_num]] = scaler.fit_transform(df[cols_num])

PLAZO = 90   # plazo contractual de construcción (días)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['dias_activa'].clip(upper=900), bins=30, color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].axvline(PLAZO, color='#ef4444', linestyle='--', label=f'{PLAZO} días (plazo)')
axes[0].set_title('Distribución de días activa'); axes[0].legend(fontsize=9)

# Duración promedio con barra de error por estado
estados = ['Iniciada','Avanzada','Finalizada','Adjudicada']
col_e   = ['#f59e0b','#3b82f6','#22c55e','#6366f1']
st_e = df.groupby('estado')['dias_activa'].agg(['mean','std']).reindex(
    [e for e in estados if e in df['estado'].values])
axes[1].bar(st_e.index, st_e['mean'], yerr=st_e['std'], capsize=5,
            color=[col_e[estados.index(e)] for e in st_e.index], alpha=0.85)
axes[1].axhline(PLAZO, color='#ef4444', linestyle='--', alpha=0.6, label=f'{PLAZO} días (plazo)')
for i, (e, row) in enumerate(st_e.iterrows()):
    axes[1].text(i, row['mean']+row['std']+5, f"{row['mean']:.0f}d", ha='center', fontsize=9)
axes[1].set_title('Días activa promedio ± desvío por estado')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('img/02_duraciones.png', bbox_inches='tight'); plt.show()
print(df.groupby('estado')['dias_activa'].agg(['mean','median','max']).round(0).astype(int))

In [ ]:
# 5. Guardar dataset procesado en Drive
output = DRIVE / 'viviendas_procesadas.csv'
df.to_csv(output, index=False)
print(f'Guardado: viviendas_procesadas.csv')
print(f'Columnas totales: {df.shape[1]} (originales: {df_raw.shape[1]} | nuevas: {df.shape[1]-df_raw.shape[1]})')
print('\nColumnas nuevas:', [c for c in df.columns if c not in df_raw.columns])